Setup

In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd
from elexon import fetch_mid, fetch_wind, fetch_demand

START, END = "2026-08-01T00:00Z", "2026-10-01T00:00Z"

Calls

In [2]:
REFRESH = False

if REFRESH:
    fetch_mid(START, END).to_csv("data/mid.csv")
    fetch_wind(START, END).to_csv("data/wind.csv")
    fetch_demand(START, END).to_csv("data/demand.csv")

mid = pd.read_csv("data/mid.csv", index_col="startTime", parse_dates=True)
wind = pd.read_csv("data/wind.csv", index_col="startTime", parse_dates=True)
demand = pd.read_csv("data/demand.csv", index_col="startTime", parse_dates=True)

Clean and align: inner join on UTC start time

In [3]:
df = pd.concat([mid, wind, demand], axis = 1, join = "inner", sort = True)
local = df.index.tz_convert("Europe/London")
df["hour"] = local.hour
df["weekday"] = local.day_of_week      # 0 = Monday
df["is_weekend"] = df["weekday"] >= 5
assert df.notna().all().all(), "NaNs after merge"
df.to_csv("data/merged.csv")